# Ranking Vendor
Rekomendasi vendor untuk kebutuhan toko souvenir/hampers pernikahan, dilakukan dalam dua langkah:
1. **Filtering** berdasarkan input user website: kategori, sub-kategori, tipe vendor (opsional), jumlah pesanan, budget, dan sisa hari menuju acara.
2. **Skoring** kandidat yang lolos filter, dengan menggabungkan risiko vendor (model notebook 03–05), harga, dan lead time.

Fitur item (harga, MOQ, kapasitas) hanya dipakai di sini dan tidak masuk model risiko.

# Import Library

In [1]:
import os
import sys
import json
import pandas as pd
import joblib

sys.path.append('..')
from app.risk_scoring import assign_risk_tier

# Load Artefak & Data

In [2]:
bundle = joblib.load('../models/v1/vendai_model.pkl')
config = bundle['risk_tier_config']

vendors = pd.read_csv('../data/processed/supplier_clean.csv')
catalog = pd.read_csv('../data/processed/vendor_catalog.csv')
print(len(vendors), 'vendor |', len(catalog), 'item')

24112 vendor | 36065 item


# Risk Score per Vendor
Risk tier dihitung **sekali per vendor** (batch), lalu di-join ke katalog. Tujuannya agar ranking tidak memanggil model berulang kali per request. `Lead_Time_Days` yang kosong diisi median train (`tech_medians`), sama seperti di faktor teknis.

In [3]:
risk = assign_risk_tier(bundle['pipeline'], vendors, config)
vendor_risk = pd.concat([vendors[['Supplier_ID', 'Lead_Time_Days']], risk], axis=1)
vendor_risk['Lead_Time_Days'] = vendor_risk['Lead_Time_Days'].fillna(config['tech_medians']['Lead_Time_Days'])

scored = catalog.merge(vendor_risk, on='Supplier_ID', how='left')
assert scored['Risk_Score'].notna().all(), 'Ada item tanpa risk score'
scored.head()

c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\HP\anaconda3\envs\capstone_project\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\HP\anaconda3\envs\capstone_pr

,Supplier_ID,Vendor_Type,Main_Category,Sub_Category,Item_Description,Unit,Unit_Price_IDR,MOQ,Capacity_Per_Week,Capacity_Per_Month,Lead_Time_Days,Risk_Score,Operasional_Buruk,Teknis_Buruk,Kriteria_Teknis_Gagal,Risk_Tier
0,SUP_30248,Home Industry,Packaging,Kartu Ucapan & Label,"Kartu ucapan terima kasih art carton 260 gsm, ...",pcs,1900,25,3830,15320,9.0,0.042920,False,False,Previous_Disruptions,Rendah
1,SUP_23186,Produsen/Pabrik,Kustomisasi & Produksi,Sablon/DTF T-Shirt,"Jasa sablon rubber kaos custom ukuran S-XXL, g...",pcs,84500,50,2480,9920,3.0,0.975397,True,False,Defect_Rate,Sedang
2,SUP_29736,Konveksi UMKM,Kustomisasi & Produksi,Produksi Handuk Custom,Produksi handuk 50x100 cm dengan label tenun n...,pcs,28500,50,680,2720,14.0,0.994200,True,True,"Lead_Time_Days, Previous_Disruptions",Tinggi
3,SUP_27251,Distributor/Grosir,Bahan Baku,Dakron/Isi Bantal,"Dakron premium untuk isi bantal, kemasan 10 kg",kg,31000,5,355,1420,16.0,0.122074,False,True,"On_Time_Delivery_Rate, Lead_Time_Days",Sedang
4,SUP_25660,Distributor/Grosir,Bahan Baku,Kain Terry/Handuk,"Kain terry microfiber, gramasi 450 gsm, cocok ...",kg,85000,5,568,2270,9.0,0.044295,False,False,,Rendah


# Fungsi Ranking
**Filter** (semua harus terpenuhi):

| Kriteria | Aturan |
|---|---|
| Kategori | `Main_Category` dan `Sub_Category` sesuai pilihan user; `Vendor_Type` opsional |
| MOQ | `MOQ ≤ quantity` |
| Budget | `Unit_Price_IDR × quantity ≤ budget` |
| Waktu | `Lead_Time_Days ≤ deadline_days` (sisa hari menuju acara) |
| Kapasitas | `quantity ≤ Capacity_Per_Week × (deadline_days − Lead_Time_Days) / 7`. Lead time dianggap waktu antre + kirim, dan sisa waktunya untuk produksi. Tanpa deadline: `quantity ≤ Capacity_Per_Month` |
| Risiko | `Risk_Tier` bukan `Tinggi` |

**Skor** = `0.5·(1 − Risk_Score) + 0.3·skor_harga + 0.2·skor_lead_time`. Skor harga dan lead time di-min-max di antara kandidat (termurah/tercepat = 1), sehingga harga hanya dibandingkan dalam sub-kategori yang sama.

In [4]:
RANKING_CONFIG = {
    'weights': {'risk': 0.5, 'price': 0.3, 'lead_time': 0.2},
    'exclude_tiers': ['Tinggi'],
}
OUTPUT_COLS = ['Rank', 'Supplier_ID', 'Vendor_Type', 'Item_Description', 'Unit', 'Unit_Price_IDR', 'MOQ',
               'Total_Price', 'Lead_Time_Days', 'Capacity_Per_Week', 'Risk_Score', 'Risk_Tier',
               'Kriteria_Teknis_Gagal', 'Score']


def inverse_minmax(s):
    """Nilai terkecil = 1, terbesar = 0. Jika semua sama, semuanya 1."""
    span = s.max() - s.min()
    return pd.Series(1.0, index=s.index) if span == 0 else (s.max() - s) / span


def rank_vendors(catalog, main_category, sub_category, quantity, budget,
                 deadline_days=None, vendor_type=None, top_k=5, config=RANKING_CONFIG):
    c = catalog[(catalog['Main_Category'] == main_category)
                & (catalog['Sub_Category'] == sub_category)
                & (catalog['MOQ'] <= quantity)
                & (catalog['Unit_Price_IDR'] * quantity <= budget)
                & ~catalog['Risk_Tier'].isin(config['exclude_tiers'])]
    if vendor_type is not None:
        c = c[c['Vendor_Type'] == vendor_type]
    if deadline_days is None:
        c = c[quantity <= c['Capacity_Per_Month']]
    else:
        production_days = deadline_days - c['Lead_Time_Days']
        c = c[(production_days >= 0) & (quantity <= c['Capacity_Per_Week'] * production_days / 7)]

    w = config['weights']
    c = c.assign(
        Total_Price=c['Unit_Price_IDR'] * quantity,
        Score=w['risk'] * (1 - c['Risk_Score'])
              + w['price'] * inverse_minmax(c['Unit_Price_IDR'])
              + w['lead_time'] * inverse_minmax(c['Lead_Time_Days']),
    )
    top = c.sort_values('Score', ascending=False).head(top_k)
    top.insert(0, 'Rank', range(1, len(top) + 1))
    return top[OUTPUT_COLS].reset_index(drop=True)

# Simulasi Kebutuhan Toko
Tiga skenario dari rantai pasok hampers pernikahan: jasa produksi, bahan baku, dan packaging. Setiap hasil dicek dengan `assert`: tidak kosong, memenuhi semua filter, dan terurut menurun berdasarkan skor.

In [5]:
def check(result, request):
    assert len(result) > 0, f'Tidak ada vendor untuk {request}'
    q, d = request['quantity'], request.get('deadline_days')
    assert (result['MOQ'] <= q).all()
    assert (result['Total_Price'] <= request['budget']).all()
    assert (result['Risk_Tier'] != 'Tinggi').all()
    if d is not None:
        assert (result['Lead_Time_Days'] <= d).all()
        assert (q <= result['Capacity_Per_Week'] * (d - result['Lead_Time_Days']) / 7).all()
    assert result['Score'].is_monotonic_decreasing


scenarios = {
    'Produksi handuk 300 pcs': dict(main_category='Kustomisasi & Produksi', sub_category='Produksi Handuk Custom',
                                    quantity=300, budget=12_000_000, deadline_days=21),
    'Bahan kain katun 50 kg': dict(main_category='Bahan Baku', sub_category='Kain Katun Combed',
                                   quantity=50, budget=7_500_000, deadline_days=14),
    'Box hampers 500 pcs': dict(main_category='Packaging', sub_category='Box Hampers',
                                quantity=500, budget=10_000_000, deadline_days=30),
}

results = {}
for name, req in scenarios.items():
    results[name] = rank_vendors(scored, **req)
    check(results[name], req)
print('Semua skenario valid.')

Semua skenario valid.


In [6]:
results['Produksi handuk 300 pcs']

,Rank,Supplier_ID,Vendor_Type,Item_Description,Unit,Unit_Price_IDR,MOQ,Total_Price,Lead_Time_Days,Capacity_Per_Week,Risk_Score,Risk_Tier,Kriteria_Teknis_Gagal,Score
0,1,SUP_02808,Produsen/Pabrik,Produksi handuk 30x30 cm dengan label tenun na...,pcs,12500,50,3750000,3.0,7930,0.043931,Rendah,Defect_Rate,0.957696
1,2,SUP_05059,Konveksi UMKM,Produksi handuk 30x30 cm dengan label tenun na...,pcs,13500,50,4050000,3.0,908,0.031361,Sedang,"On_Time_Delivery_Rate, Previous_Disruptions",0.953811
2,3,SUP_17194,Konveksi UMKM,Produksi handuk 30x70 cm dengan print nama pen...,pcs,13500,200,4050000,3.0,655,0.036667,Rendah,,0.951158
3,4,SUP_20340,Konveksi UMKM,Produksi handuk 50x100 cm dengan print nama pe...,pcs,14500,25,4350000,3.0,550,0.021445,Rendah,,0.948599
4,5,SUP_06064,Produsen/Pabrik,Produksi handuk 50x100 cm dengan print nama pe...,pcs,15000,50,4500000,3.0,3410,0.014111,Rendah,,0.947182


In [7]:
results['Bahan kain katun 50 kg']

,Rank,Supplier_ID,Vendor_Type,Item_Description,Unit,Unit_Price_IDR,MOQ,Total_Price,Lead_Time_Days,Capacity_Per_Week,Risk_Score,Risk_Tier,Kriteria_Teknis_Gagal,Score
0,1,SUP_28279,Produsen/Pabrik,"Kain katun combed 30s untuk kaos sablon, terse...",kg,106500,50,5325000,3.0,762,0.030724,Rendah,Defect_Rate,0.911594
1,2,SUP_32333,Produsen/Pabrik,"Kain katun combed 20s untuk kaos sablon, terse...",kg,107500,10,5375000,3.0,910,0.052556,Rendah,Defect_Rate,0.895461
2,3,SUP_25441,Distributor/Grosir,"Kain katun combed 30s untuk kaos sablon, terse...",kg,110500,25,5525000,3.0,555,0.058660,Rendah,,0.876757
3,4,SUP_07324,Produsen/Pabrik,"Kain katun combed 20s untuk kaos sablon, terse...",kg,111000,25,5550000,3.0,662,0.056212,Rendah,On_Time_Delivery_Rate,0.875372
4,5,SUP_24357,Distributor/Grosir,"Kain katun combed 24s untuk kaos sablon, terse...",kg,110000,5,5500000,3.0,322,0.088982,Rendah,On_Time_Delivery_Rate,0.864205


In [8]:
results['Box hampers 500 pcs']

,Rank,Supplier_ID,Vendor_Type,Item_Description,Unit,Unit_Price_IDR,MOQ,Total_Price,Lead_Time_Days,Capacity_Per_Week,Risk_Score,Risk_Tier,Kriteria_Teknis_Gagal,Score
0,1,SUP_18985,Konveksi UMKM,Box hampers kraft ukuran besar dengan sablon nama,pcs,8000,25,4000000,5.0,778,0.031055,Sedang,"On_Time_Delivery_Rate, Defect_Rate",0.953425
1,2,SUP_12788,Supplier Packaging,Box hampers hardbox ukuran kecil dengan sekat ...,pcs,8500,200,4250000,3.0,1150,0.078958,Rendah,On_Time_Delivery_Rate,0.936521
2,3,SUP_03192,Supplier Packaging,Box hampers hardbox ukuran sedang dengan sekat...,pcs,8000,50,4000000,3.0,1335,0.118209,Rendah,Defect_Rate,0.928895
3,4,SUP_33160,Supplier Packaging,Box hampers hardbox ukuran sedang dengan sekat...,pcs,8500,200,4250000,7.0,2410,0.053514,Rendah,,0.911148
4,5,SUP_12601,Supplier Packaging,Box hampers kraft ukuran kecil dengan tutup ma...,pcs,8500,25,4250000,8.0,3330,0.047763,Sedang,"On_Time_Delivery_Rate, Previous_Disruptions",0.904499


## Filter Opsional: Tipe Vendor

In [9]:
req = dict(scenarios['Produksi handuk 300 pcs'], vendor_type='Home Industry')
home = rank_vendors(scored, **req)
check(home, req)
assert (home['Vendor_Type'] == 'Home Industry').all()
home

,Rank,Supplier_ID,Vendor_Type,Item_Description,Unit,Unit_Price_IDR,MOQ,Total_Price,Lead_Time_Days,Capacity_Per_Week,Risk_Score,Risk_Tier,Kriteria_Teknis_Gagal,Score
0,1,SUP_25989,Home Industry,Produksi handuk 50x100 cm dengan print nama pe...,pcs,12500,200,3750000,3.0,510,0.090107,Rendah,On_Time_Delivery_Rate,0.934608
1,2,SUP_11256,Home Industry,Produksi handuk 30x70 cm dengan print nama pen...,pcs,14000,50,4200000,6.0,1500,0.015066,Rendah,,0.916874
2,3,SUP_05179,Home Industry,Produksi handuk 30x70 cm dengan label tenun na...,pcs,15500,50,4650000,5.0,398,0.021634,Rendah,,0.911669
3,4,SUP_29400,Home Industry,Produksi handuk 50x100 cm dengan print nama pe...,pcs,15500,100,4650000,5.0,328,0.045067,Rendah,,0.899953
4,5,SUP_30243,Home Industry,Produksi handuk 50x100 cm dengan print nama pe...,pcs,17500,25,5250000,4.0,200,0.035577,Rendah,,0.897692


# Simpan Artefak Ranking
- `vendor_catalog_scored.csv`: katalog + risk score/tier per item, siap dipakai untuk filtering.
- `ranking_config.json`: bobot skor dan tier yang dikecualikan.

In [10]:
scored.to_csv('../models/v1/vendor_catalog_scored.csv', index=False)

with open('../models/v1/ranking_config.json', 'w') as f:
    json.dump(RANKING_CONFIG, f, indent=2)

os.listdir('../models/v1')

['model_pipeline.pkl',
 'ranking_config.json',
 'risk_tier_config.json',
 'vendai_model.pkl',
 'vendor_catalog_scored.csv']